# Merge Four NileTTS Train QA CSV Files

Use this notebook after all 4 teammates finish their chunks.

Expected files:

- `niletts_llm_qa_train_part_0.csv`
- `niletts_llm_qa_train_part_1.csv`
- `niletts_llm_qa_train_part_2.csv`
- `niletts_llm_qa_train_part_3.csv`

Final output:

- `niletts_llm_qa_train_merged_filtered.csv`

### ======
### Earlier merge version

This notebook merges and filters the four training QA parts.

The final pipeline uses `6_checked_merge_niletts_train_qa_parts.ipynb`,
which additionally normalizes question-type and difficulty labels
and preserves their original values in raw-label columns.
### ======

In [ ]:
%pip install "numpy==1.26.4" "pandas==2.2.2"

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42


In [ ]:
# =====
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "2_QA_GENERATION").is_dir():
        QA_DIR = candidate / "2_QA_GENERATION"
        break
else:
    raise FileNotFoundError(
        "Open the notebook from inside the project folder."
    )
# =====

In [ ]:
# =====
# part_files = [
#     "niletts_llm_qa_train_part_0.csv",
#     "niletts_llm_qa_train_part_1.csv",
#     "niletts_llm_qa_train_part_2.csv",
#     "niletts_llm_qa_train_part_3.csv",
# ]

part_files = [
    QA_DIR / "1_train_data" / "filtered"
    / f"niletts_llm_qa_train_part_{part_id}.csv"
    for part_id in range(4)
]
# =====

missing = [f for f in part_files if not Path(f).exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

dfs = []

for f in part_files:
    df = pd.read_csv(f)
    print(f, df.shape)
    dfs.append(df)

merged_df = pd.concat(dfs, ignore_index=True)

print("Merged raw shape:", merged_df.shape)
display(merged_df.head())


In [ ]:
required_cols = [
    "id", "split", "part_id", "metadata_row_index", "qa_train_source_index",
    "audio_file", "speaker_name", "domain", "context_ref", "question",
    "answer", "answer_start_ref", "valid_answer_span_ref", "question_type", "difficulty"
]

missing_cols = [c for c in required_cols if c not in merged_df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in merged data: {missing_cols}")

print("All required columns exist.")


In [ ]:
def normalize_spaces(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def find_answer_start_soft(context, answer):
    context = str(context)
    answer = str(answer)

    exact = context.find(answer)
    if exact != -1:
        return exact

    context_norm = normalize_spaces(context)
    answer_norm = normalize_spaces(answer)

    norm_pos = context_norm.find(answer_norm)
    if norm_pos != -1:
        return norm_pos

    return -1


clean_df = merged_df.copy()

for col in ["question", "answer", "context_ref", "audio_file", "speaker_name", "domain"]:
    clean_df[col] = clean_df[col].astype(str).map(normalize_spaces)

clean_df["answer_start_ref"] = clean_df.apply(
    lambda r: find_answer_start_soft(r["context_ref"], r["answer"]),
    axis=1
)

clean_df["valid_answer_span_ref"] = clean_df["answer_start_ref"] != -1

before = len(clean_df)

clean_df = clean_df[clean_df["question"].str.len() > 5]
clean_df = clean_df[clean_df["answer"].str.len() > 0]
clean_df = clean_df[clean_df["valid_answer_span_ref"] == True]

clean_df = clean_df.drop_duplicates(
    subset=["audio_file", "question", "answer"]
).reset_index(drop=True)

print("Rows before cleaning:", before)
print("Rows after cleaning:", len(clean_df))


In [ ]:
expected_train_rows = 7713

covered_source_rows = clean_df["qa_train_source_index"].nunique()
missing_count = expected_train_rows - covered_source_rows

print("Expected QA train source rows:", expected_train_rows)
print("Covered source rows:", covered_source_rows)
print("Missing source rows:", missing_count)

covered_indices = set(clean_df["qa_train_source_index"].astype(int).tolist())
missing_indices = sorted(list(set(range(expected_train_rows)) - covered_indices))

print("First 30 missing source indices:")
print(missing_indices[:30])

coverage_report = pd.DataFrame({
    "metric": [
        "expected_train_source_rows",
        "covered_train_source_rows",
        "missing_train_source_rows",
        "total_qa_pairs_after_cleaning"
    ],
    "value": [
        expected_train_rows,
        covered_source_rows,
        missing_count,
        len(clean_df)
    ]
})

display(coverage_report)


In [ ]:
print("Rows per part:")
display(clean_df["part_id"].value_counts().sort_index())

print("QA pairs per domain:")
display(clean_df["domain"].value_counts())

print("QA pairs per question type:")
display(clean_df["question_type"].value_counts(dropna=False))

print("QA pairs per difficulty:")
display(clean_df["difficulty"].value_counts(dropna=False))

print("Average QA pairs per covered transcript:")
print(len(clean_df) / max(1, clean_df["qa_train_source_index"].nunique()))


In [ ]:
final_df = clean_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# =====
# final_output = "niletts_llm_qa_train_merged_filtered.csv"
final_output = (
    QA_DIR / "1_train_data"
    / "niletts_llm_qa_train_merged_filtered.csv"
)
# =====

final_df.to_csv(final_output, index=False, encoding="utf-8-sig")

print("Saved:", final_output)
print("Final shape:", final_df.shape)

display(final_df.head())


In [ ]:
missing_df = pd.DataFrame({"qa_train_source_index": missing_indices})

# =====
missing_output = (
    QA_DIR / "1_train_data" / "missing"
    / "missing_qa_train_source_indices.csv"
)

# missing_df.to_csv("missing_qa_train_source_indices.csv", index=False, encoding="utf-8-sig")
missing_df.to_csv(missing_output, index=False, encoding="utf-8-sig"
)

# print("Saved: missing_qa_train_source_indices.csv")
print("Saved:", missing_output)
# =====

display(missing_df.head())
